In [1]:
import sys
sys.path.append('/home/fkunneman/scripts/')

In [2]:
import gc
import importlib
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent

In [3]:
### Load models for chatbot and rag in memory

os.environ["HF_TOKEN"] = 'YOUR_TOKEN_HERE'
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=150, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.7 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda


In [4]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="instruction_db",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [ ]:

instructions = '/data/opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
pat = '/data/opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa = '/data/opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v3.csv'
nav = '/data/opslag_inclusieve_spraakassistent_project/navigation.csv'

importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm)
assistant.prepare_instructions(instructions)
assistant.prepare_patterns(pat)
assistant.setup_rag(collection,qa,nav)

Instruct ["Waar begint je reis? Vul dat in bij 'Van'.", "Waar wil je naartoe? Vul dat in bij 'Naar'.", "Wanneer wil je vertrekken? 'nu' of 'later'?", "Klik dan op de knop 'vertrek'", "Als je op en latere dag wil reizen,: klik dan op de knop 'datum' en kies de dag.", "Klik op de knop 'tijd'. Kies hoe laat je wilt vertrekken.", "Klik op de blauwe knop 'Plan je reis'. Je ziet nu de informatie over de ov-rit in beeld."]
PAT ["Hallo! Ik kan je helpen met 'een reis plannen' of 'een afspraak maken voor een paspoort'. Je kan 'vorige' zeggen als je een stap terug wilt doen,: 'volgende' als je naar de volgende stap wilt. Zeg 'herhaal' als je wilt dat ik een stap herhaal. Je kan altijd een vraag stellen. Zeg nu 'reis' of 'paspoort'.", "Heb je hier nog vragen? Stel ze gerust. Geen vragen? Zeg dan 'klaar'"]


In [14]:
assistant.chat()

ConversationBufferMemory initialized and current_instruction_index set to 0.
Start interacting with the conversational agent. Type 'quit' or 'exit' to end the conversation.


You:  ov


Agent: Hallo! Ik kan je helpen met 'een reis plannen' of 'een afspraak maken voor een paspoort'. Je kan 'vorige' zeggen als je een stap terug wilt doen
None: 'volgende' als je naar de volgende stap wilt. Zeg 'herhaal' als je wilt dat ik een stap herhaal. Je kan altijd een vraag stellen. Zeg nu 'reis' of 'paspoort'.


You:  quit


Agent: Tot ziens!
